# `src/` usage tour

Walkthrough of every public function in the shared `src/` package. Run top-to-bottom; each section is self-contained.

**Modules**
- `src.data` — bundle loader (`load_bundle`, `Bundle`)
- `src.splits` — split selectors (`train_positives`, `val_pairs`, `test_pairs`, `eval_users`, `train_user_items`)
- `src.candidates` — candidate utilities (`filter_seen`, `popularity_fallback`, `merge_topk`, `topk_dataframe`)
- `src.eval` — metrics (`evaluate`, `evaluate_by_tier`, `rating_metrics`)
- `src.io` — predictions I/O (`save_predictions`, `load_predictions`, `list_predictions`)

**Prediction-file contract everyone follows:** `(user_id:int32, item_id:int32, rank:int32, score:float32)` written to `predictions/{model}_topk.parquet`.

**Column-naming note:** the on-disk parquets use `uid`/`iid`; `load_bundle` renames those to `user_id`/`item_id` in memory. Every DataFrame on `Bundle` uses the new names.

## 0. Setup

The notebook lives at `<repo>/notebooks/`. Add the parent (the repo root) to `sys.path` so `import src` works.

In [1]:
import sys, time
from pathlib import Path

# Make `src` importable from this notebook.
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd

import src
print(dir(src))

['Bundle', 'PRED_COLUMNS', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'candidates', 'data', 'eval', 'eval_users', 'evaluate', 'evaluate_by_tier', 'filter_seen', 'io', 'load_bundle', 'load_predictions', 'merge_topk', 'popularity_fallback', 'rating_metrics', 'save_predictions', 'splits', 'test_pairs', 'train_positives', 'train_user_items', 'val_pairs']


## 1. `load_bundle()` — open the preprocessed_v1 deliverable

Returns a `Bundle` dataclass holding every table, embedding, FAISS index, vocab, and id↔row lookup. Embeddings are mmapped so memory cost is small.

**Flags**
- `load_interactions=False` skips the 34M-row `interactions_all` (~1.5 GB) — use when you only need `split.parquet`.
- `load_faiss=False` skips reading the FAISS index — content notebook needs it; baselines/ALS don't.

**Default path:** `recsys_data_v1/preprocessed_v1/` (resolved relative to the `src/` package, so you can call from anywhere).

In [2]:
t0 = time.time()
b = src.load_bundle(load_interactions=False, load_faiss=False)
print(f"loaded in {time.time()-t0:.1f}s")
print(f"n_items={b.n_items}  n_users_kcore={b.n_users_kcore}")
print(f"bundle path: {b.path}")

loaded in 0.6s
n_items=34916  n_users_kcore=380782
bundle path: /Users/ruhanirekhi/Documents/GitHub/GoodReads-YoungAdult-RecSys/recsys_data_v1/preprocessed_v1


### What's on the `Bundle`?
Every attribute is either a DataFrame, a numpy array, a FAISS index, a dict, or a small int.

In [3]:
# Structured features
print("book_features:", b.book_features.shape, list(b.book_features.columns)[:8], "...")
print("user_features:", b.user_features.shape, list(b.user_features.columns)[:8], "...")
print("item_tags    :", b.item_tags.shape, list(b.item_tags.columns))
print("user_tags    :", b.user_tags.shape, list(b.user_tags.columns))

# Interactions / labels
print("split        :", b.split.shape, "value_counts:", b.split['split'].value_counts().to_dict())
print("popularity   :", b.popularity.shape, list(b.popularity.columns))

book_features: (34916, 38) ['item_id', 'book_id', 'title', 'language_code', 'is_ebook', 'format', 'average_rating', 'log_ratings_count'] ...
user_features: (580139, 29) ['user_id', 'n_pos', 'n_read', 'n_rated', 'avg_rating', 'rating_std', 'active_days', 'first_ts'] ...
item_tags    : (32243, 6) ['item_id', 'tag_ids', 'content_warning_ids', 'style_ids', 'audience', 'n_tags']
user_tags    : (40314, 6) ['user_id', 'tag_ids', 'dealbreaker_ids', 'reading_level', 'rating_bias', 'n_tags']
split        : (15195168, 3) value_counts: {'train': 14433604, 'test': 380782, 'val': 380782}
popularity   : (34916, 7) ['item_id', 'n_interactions', 'n_positive', 'n_rated', 'avg_rating', 'pop_rank', 'pop_tier']


In [4]:
# Dense embeddings (mmapped float32)
print("desc_emb         :", b.desc_emb.shape, b.desc_emb.dtype)
print("user_content_emb :", b.user_content_emb.shape)
print("item_profile_emb :", b.item_profile_emb.shape)
print("user_profile_emb :", b.user_profile_emb.shape)

# Vocabs
print("tag_vocab keys   :", list(b.tag_vocab.keys()))
print("cat_vocab keys   :", list(b.cat_vocab.keys())[:6], "...")

desc_emb         : (34916, 384) float32
user_content_emb : (580139, 384)
item_profile_emb : (32243, 384)
user_profile_emb : (40314, 384)
tag_vocab keys   : ['tag2id', 'id2tag', 'freq', 'n_tags']
cat_vocab keys   : ['genre_top', 'language_norm', 'pop_tier', 'activity_tier', 'top_genre', 'audience'] ...


### Row ↔ id lookups
Embeddings are stored as `[N, D]` arrays. To look up a specific user or item's vector, go through the prebuilt dicts on the Bundle.

In [5]:
# Pick any item_id that has a desc embedding (all 34,916 do)
item_id = int(b.book_features['item_id'].iloc[0])
row = b.itemid_to_row_desc[item_id]
vec = b.desc_emb[row]
print(f"item_id={item_id} -> desc_emb row {row}, vector shape={vec.shape}, |v|={np.linalg.norm(vec):.4f}")

# Coverage helpers — not every user has an LLM profile
user_id = int(b.user_features['user_id'].iloc[0])
print(f"user_id={user_id}: has user_content_emb={b.has_user_content_emb(user_id)}, has user_profile_emb={b.has_user_profile_emb(user_id)}")

item_id=33692 -> desc_emb row 33692, vector shape=(384,), |v|=1.0000
user_id=24675: has user_content_emb=True, has user_profile_emb=False


### Coverage counts (recomputed from loaded data)

`Bundle` exposes four counts. The two `*_with_profile` numbers are counted from the **actual loaded index parquets**, not the manifest — so they stay accurate when LLM artifacts get swapped in place during iteration.

In [6]:
print(f"n_items              : {b.n_items:,}")
print(f"n_users_kcore        : {b.n_users_kcore:,}")
print(f"n_items_with_profile : {b.n_items_with_profile:,}  ({100*b.n_items_with_profile/b.n_items:.1f}% coverage)")
print(f"n_users_with_profile : {b.n_users_with_profile:,}  ({100*b.n_users_with_profile/b.n_users_kcore:.1f}% coverage)")

n_items              : 34,916
n_users_kcore        : 380,782
n_items_with_profile : 32,243  (92.3% coverage)
n_users_with_profile : 40,314  (10.6% coverage)


### Optional LLM artifacts (warn-on-missing)

Every LLM-derived artifact (`item_tags`, `user_tags`, `item_profile_emb` + index, `user_profile_emb` + index, `tag_vocab`) is **optional**. If a file is missing — typically mid-rerun when Huanjie is regenerating it — `load_bundle` emits a warning and sets the attribute to `None` rather than crashing.

Below we simulate a missing `user_tags.parquet` by overriding to a path that doesn't exist.

In [7]:
import warnings

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    b_partial = src.load_bundle(
        load_interactions=False, load_faiss=False,
        user_tags_path='/tmp/does_not_exist_user_tags.parquet',
    )
    for w in caught:
        print('WARN:', w.message)

print()
print(f'b_partial.user_tags is None : {b_partial.user_tags is None}')
print(f'b_partial.item_tags shape   : {b_partial.item_tags.shape}  (still loaded)')
print(f'has_user_profile_emb(0)     : {b_partial.has_user_profile_emb(0)}  (helper is None-safe)')

WARN: [load_bundle] optional artifact missing: user_tags at /tmp/does_not_exist_user_tags.parquet — proceeding with None

b_partial.user_tags is None : True
b_partial.item_tags shape   : (32243, 6)  (still loaded)
has_user_profile_emb(0)     : False  (helper is None-safe)


### Per-artifact overrides — iterate on LLM outputs without re-bundling

When the LLM owner regenerates one artifact (smaller model, fixed-length profiles, expanded user coverage), drop the new file anywhere and point `load_bundle` at it. The rest of the bundle is unchanged — no need to copy the 500MB of non-LLM artifacts.

Available overrides:
- `item_tags_path`, `user_tags_path`
- `item_profile_emb_path` + `item_profile_emb_index_path`
- `user_profile_emb_path` + `user_profile_emb_index_path`
- `tag_vocab_path`

Below we create a dummy alternate `user_profile_emb` pair to show the wiring.

In [8]:
exp_dir = Path('/tmp/llm_v2_demo')
exp_dir.mkdir(exist_ok=True)

# Dummy alternate pair — in practice this is what your re-run pipeline writes.
np.save(exp_dir / 'user_profile_emb_v2.npy', np.zeros((5, 384), dtype=np.float32))
pd.DataFrame({'row': [0,1,2,3,4], 'uid': [100,101,102,103,104]}) \
    .to_parquet(exp_dir / 'user_profile_emb_v2_index.parquet')

b_v2 = src.load_bundle(
    load_interactions=False, load_faiss=False,
    user_profile_emb_path=exp_dir / 'user_profile_emb_v2.npy',
    user_profile_emb_index_path=exp_dir / 'user_profile_emb_v2_index.parquet',
)
print(f'overridden user_profile_emb : shape={b_v2.user_profile_emb.shape}, n_users_with_profile={b_v2.n_users_with_profile}')
print(f'has_user_profile_emb(100)   : {b_v2.has_user_profile_emb(100)}')
print(f'item_profile_emb untouched  : shape={b_v2.item_profile_emb.shape}')

# A/B side-by-side: just load twice with different overrides.
print(f'\nA/B comparison')
print(f'  default bundle : {b.n_users_with_profile:,} users with LLM profile')
print(f'  experiment v2  : {b_v2.n_users_with_profile:,} users with LLM profile')

import shutil; shutil.rmtree(exp_dir)

overridden user_profile_emb : shape=(5, 384), n_users_with_profile=5
has_user_profile_emb(100)   : True
item_profile_emb untouched  : shape=(32243, 384)

A/B comparison
  default bundle : 40,314 users with LLM profile
  experiment v2  : 5 users with LLM profile


### `extras` — slot in new artifact types without editing `data.py`

Pass `{name: artifact}` through `extras=`. Useful when the LLM owner adds a new artifact type (per-profile confidence, a separate dealbreaker embedding, an audience-tag table) and wants to thread it into model notebooks without me having to extend the `Bundle` dataclass.

In [9]:
b_with_extras = src.load_bundle(
    load_interactions=False, load_faiss=False,
    extras={
        'profile_confidence': pd.DataFrame({'user_id': [0, 1, 2], 'conf': [0.91, 0.74, 0.55]}),
        'dealbreaker_emb'   : np.zeros((3, 128), dtype=np.float32),
    },
)
print('extras keys :', list(b_with_extras.extras.keys()))
print()
print('profile_confidence:')
print(b_with_extras.extras['profile_confidence'])
print('\ndealbreaker_emb shape:', b_with_extras.extras['dealbreaker_emb'].shape)

extras keys : ['profile_confidence', 'dealbreaker_emb']

profile_confidence:
   user_id  conf
0        0  0.91
1        1  0.74
2        2  0.55

dealbreaker_emb shape: (3, 128)


## 2. `src.splits` — train / val / test selectors

Every model filters through these so the held-out user set is the same everywhere.

In [10]:
tr = src.train_positives(b)
va = src.val_pairs(b)
te = src.test_pairs(b)
print(f"train pairs: {len(tr):,}")
print(f"val   pairs: {len(va):,}")
print(f"test  pairs: {len(te):,}")
tr.head(3)

train pairs: 14,433,604
val   pairs: 380,782
test  pairs: 380,782


,user_id,item_id
0,299235,3351
1,299235,7064
2,299235,4944


In [11]:
# Users with >=1 held-out test item. ONLY these contribute to ranking metrics.
eval_uids = src.eval_users(b, split='test')
print(f"eval users (test): {len(eval_uids):,}")
print(f"first 5 uids     : {eval_uids[:5]}")

eval users (test): 380,782
first 5 uids     : [0 2 5 6 7]


In [12]:
# {uid: np.ndarray of train iids} — used for seen-masking at recommend time.
# This builds a ~14M-entry dict; takes a few seconds.
t0 = time.time()
user_items = src.train_user_items(b)
print(f"built in {time.time()-t0:.1f}s, {len(user_items):,} users")

sample_uid = eval_uids[0]
print(f"uid={sample_uid} has {len(user_items.get(sample_uid, []))} train positives")

built in 6.2s, 380,782 users
uid=0 has 24 train positives


## 3. `src.candidates` — candidate-generation utilities

Helpers every model needs: drop seen items, pop fallback for cold users, merge candidate lists from multiple recall channels.

### `popularity_fallback(bundle, k, exclude=None, by='n_positive')`
Top-k iids by popularity. Used for cold users; also the building block of the popularity baseline in notebook 02.

In [13]:
top10 = src.popularity_fallback(b, k=10)
print("top 10 item_ids by n_positive:", top10.tolist())

# Decode to titles for sanity
titles = b.book_features.set_index('item_id').loc[top10, 'title']
titles

top 10 item_ids by n_positive: [17353, 18387, 31778, 5401, 34362, 1787, 18373, 16730, 12075, 17915]


item_id
17353       The Hunger Games (The Hunger Games, #1)
18387                       Twilight (Twilight, #1)
31778          Catching Fire (The Hunger Games, #2)
5401                         The Fault in Our Stars
34362             Mockingjay (The Hunger Games, #3)
1787                      Divergent (Divergent, #1)
18373                       New Moon (Twilight, #2)
16730                        Eclipse (Twilight, #3)
12075    City of Bones (The Mortal Instruments, #1)
17915                  Breaking Dawn (Twilight, #4)
Name: title, dtype: object

### `filter_seen(iids, seen, k=None)`
Drop items the user already interacted with. Forgetting this inflates Recall by 5–10× — never skip it.

Typical usage: pass the user's train positives as `seen`, then take the first `k` survivors.

In [14]:
uid = eval_uids[0]
seen = user_items.get(uid, np.empty(0, dtype=np.int64))

# Take top-50 popular, filter seen, keep top-10
candidates = src.popularity_fallback(b, k=50)
recs = src.filter_seen(candidates, seen, k=10)
print(f"uid={uid} has {len(seen)} train items; popularity top-10 after seen-mask: {recs.tolist()}")

uid=0 has 24 train items; popularity top-10 after seen-mask: [18387, 18373, 16730, 18665, 2882, 4337, 32054, 15549, 4101, 32645]


### `merge_topk(score_dicts, k=20, normalize=True)`
Weighted union of candidate→score dicts from multiple recall channels (used by the hybrid in notebook 06). Each channel is min-max scaled to [0,1] by default so ALS scores (~10) and cosine scores (~1) are comparable.

In [15]:
# Fake two channels for illustration
als_scores     = {101: 8.2, 202: 7.1, 303: 6.4, 404: 5.0}
content_scores = {202: 0.91, 303: 0.85, 505: 0.83, 606: 0.77}

merged = src.merge_topk(
    [(als_scores, 1.0), (content_scores, 0.5)],   # weight content half as much as ALS
    k=5,
)
for iid, score in merged:
    print(f"  iid={iid:>4}  merged_score={score:.3f}")

  iid= 202  merged_score=1.156
  iid= 101  merged_score=1.000
  iid= 303  merged_score=0.723
  iid= 505  merged_score=0.214
  iid= 404  merged_score=0.000


### `topk_dataframe(uid_to_ranked, k=20)`
Convenience: turn a `{uid: [iid, ...]}` (or `{uid: [(iid, score), ...]}`) dict into the standard `(uid, iid, rank, score)` DataFrame ready for `save_predictions`.

In [16]:
from src.candidates import topk_dataframe

demo = {
    1001: [(11, 0.9), (22, 0.8), (33, 0.7)],
    1002: [44, 55, 66],   # iids only; synthetic descending scores will be filled in
}
topk_dataframe(demo, k=5)

,user_id,item_id,rank,score
0,1001,11,0,0.9
1,1001,22,1,0.8
2,1001,33,2,0.7
3,1002,44,0,5.0
4,1002,55,1,4.0
5,1002,66,2,3.0


## 4. `src.eval` — ranking + rating metrics

**`evaluate(pred_df, bundle, split='test', ks=(5,10,20))`** returns recall/precision/ndcg/map at each k, averaged over `eval_users(split)`.

Users missing from `pred_df` count as 0 hits — this keeps coverage gaps honest.

Let's build a real (small) popularity baseline and evaluate it.

In [17]:
# Build popularity-top-20-per-user, with seen-masking, for ALL eval users.
# This is essentially the simplest model in notebook 02.
K = 20
pool = src.popularity_fallback(b, k=200)   # take a deep pool so seen-masking can't run out

t0 = time.time()
rows = []
for user_id in eval_uids:
    seen = user_items.get(int(user_id), np.empty(0, dtype=np.int64))
    recs = src.filter_seen(pool, seen, k=K)
    for rank, item_id in enumerate(recs):
        rows.append((int(user_id), int(item_id), rank, float(K - rank)))
pred_df = pd.DataFrame(rows, columns=['user_id', 'item_id', 'rank', 'score'])
print(f"built {len(pred_df):,} pred rows for {pred_df['user_id'].nunique():,} users in {time.time()-t0:.1f}s")
pred_df.head()

built 7,615,638 pred rows for 380,782 users in 16.2s


,user_id,item_id,rank,score
0,0,18387,0,20.0
1,0,18373,1,19.0
2,0,16730,2,18.0
3,0,18665,3,17.0
4,0,2882,4,16.0


In [18]:
metrics = src.evaluate(pred_df, b, split='test', ks=[5, 10, 20])
for k, v in metrics.items():
    print(f"  {k:<18} {v:.4f}")

  recall@5           0.0720
  recall@10          0.1118
  recall@20          0.1671
  precision@5        0.0144
  precision@10       0.0112
  precision@20       0.0084
  ndcg@5             0.0455
  ndcg@10            0.0582
  ndcg@20            0.0723
  map@5              0.0368
  map@10             0.0420
  map@20             0.0458
  n_eval_users       380782.0000


### `evaluate_by_tier` — head / mid / tail breakdown
Buckets users by the popularity tier of their held-out item. Popularity baselines should crush `head` and collapse on `tail` — that's exactly the bias we want every other model to fix.

In [19]:
tier_df = src.evaluate_by_tier(pred_df, b, split='test', ks=[10])
tier_df.pivot(index='tier', columns='metric', values='value').loc[['head','mid','tail']]

metric,map,ndcg,precision,recall
tier,,,,
head,0.068938,0.095655,0.018367,0.183675
mid,0.000000,0.000000,0.000000,0.000000
tail,0.000000,0.000000,0.000000,0.000000


### `rating_metrics` (ALS only)
RMSE/MAE on held-out ratings. Only the ALS notebook (03) calls this — it's the proposal's rating-prediction side task. Requires `load_interactions=True` because we need the actual `rating` column on the held-out edges.

**Skipping live demo** since we loaded the bundle with `load_interactions=False`. Usage:
```python
b_full = src.load_bundle()                                # interactions on
rating_df = pd.DataFrame({'user_id': [...], 'item_id': [...], 'rating_pred': [...]})
src.rating_metrics(rating_df, b_full, split='test')        # {'rmse': ..., 'mae': ..., 'n': ...}
```

## 5. `src.io` — save / load predictions

Every notebook ends by writing `predictions/{name}_topk.parquet`. `save_predictions` enforces the schema, coerces dtypes, and checks that `rank` starts at 0 per user.

In [20]:
out_path = src.save_predictions(pred_df, name='popularity_demo')
print(f"wrote {out_path}")

loaded = src.load_predictions('popularity_demo')
print("loaded shape:", loaded.shape, "dtypes:", loaded.dtypes.to_dict())

# What's currently saved?
from src.io import list_predictions
print("available models on disk:", list_predictions())

wrote /Users/ruhanirekhi/Documents/GitHub/GoodReads-YoungAdult-RecSys/predictions/popularity_demo_topk.parquet
loaded shape: (7615638, 4) dtypes: {'user_id': dtype('int32'), 'item_id': dtype('int32'), 'rank': dtype('int32'), 'score': dtype('float32')}
available models on disk: ['popularity_demo']


In [21]:
# Clean up the demo file so it doesn't get picked up by notebook 08
out_path.unlink()
print("removed demo file")

removed demo file


## 6. Putting it together — the pattern every model notebook follows

```python
import src
b = src.load_bundle()                            # 1. load bundle
eval_uids   = src.eval_users(b, 'test')          # 2. lock in eval set
user_items  = src.train_user_items(b)            # 3. seen-mask source

# 4. produce ranked candidates for each eval user (model-specific)
uid_to_recs = train_and_recommend(b, eval_uids, user_items, k=20)

# 5. dump predictions with the standard schema
pred_df = src.candidates.topk_dataframe(uid_to_recs, k=20)
src.save_predictions(pred_df, name='my_model')

# 6. report metrics (notebook 08 will re-run this for the comparison table)
print(src.evaluate(pred_df, b, split='test', ks=[5, 10, 20]))
print(src.evaluate_by_tier(pred_df, b, split='test', ks=[10]))
```

That's the contract. Any notebook that produces a `predictions/*_topk.parquet` with the right schema and the right eval-user coverage drops cleanly into 08.